# NHA Hackathon – Problem Statement 3  
## Skeletal Notebook: Document Forgery / Deepfake Detection

## Problem Understanding

According to the uploaded guidelines, PS3 requires participants to detect visible document tampering, classify each image/page using the predefined category IDs, and highlight manipulated regions using YAML annotations. The submission must contain:
1. a per-document or per-page **YAML annotation file**, and
2. a **JSON output** with `link`, `file_name`, and `Category_ID`.  

PDFs must be converted into per-page JPG-style entries such as `xyz_page_1.jpg` and produce corresponding YAML files such as `xyz_page_1.yaml`. Categories C8 and C10 are category-only cases with no YAML requirement, while other categories require bounding boxes in the exact YAML syntax expected by the organizers. fileciteturn1file0L1-L19 fileciteturn1file0L20-L30

## Categories in Scope

Use only the predefined category IDs:
- `C1` Copy-paste within the same document
- `C2` Overwriting on existing text
- `C3` Adding new content
- `C4` Removing / erasing text or image
- `C5` Merging content from different documents
- `C6` Watermark removal
- `C7` Irregular spacing
- `C8` Fully AI-generated document
- `C9` Partial AI-generated edits
- `C10` No editing / discrepancy found fileciteturn1file0L20-L30

In [1]:
!pip install -q opencv-python-headless pytesseract numpy scikit-learn scipy onnxruntime Pillow PyMuPDF PyYAML pandas

!pip install "optimum[onnxruntime]"

# If on the cloud Linux instance provided, we must install Hindi language packs for C2/C7 we will uncomment it while seding to the cloud Noteboollm
# !sudo apt-get install tesseract-ocr-hin -y

import os
import io
import json
import math
import asyncio
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
import pandas as pd

import cv2
import numpy as np
import pytesseract
import scipy.fftpack
import yaml

try:
    from PIL import Image, ImageChops, ImageEnhance
except ImportError:
    Image = None

try:
    import fitz  # PyMuPDF for PDF conversion
except ImportError:
    fitz = None

try:
    import onnxruntime as ort # Staged for Tier 4 (Optional AI)
except ImportError:
    ort = None

# IMPORTANT: If testing on a local Windows machine, uncomment and set your Tesseract path.
# pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

# Define standard paths and Category IDs per hackathon rules
INPUT_DIR = Path("./Claim_Documents")   #create the dir & store you data init
OUTPUT_DIR = Path("./outputs")
ANNOTATIONS_DIR = OUTPUT_DIR / "annotations"
DEBUG_DIR = OUTPUT_DIR / "rendered_pages"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ANNOTATIONS_DIR.mkdir(parents=True, exist_ok=True)
DEBUG_DIR.mkdir(parents=True, exist_ok=True)

SUPPORTED_EXTS = {".jpg", ".jpeg", ".png", ".pdf"}

print("Environment Setup and Imports Successful.")

Environment Setup and Imports Successful.


## Download the Dataset
We have provided a dedicated widget to download the hackathon datasets directly from the platform into this notebook environment.

### 1. Import the Widget

from databank_download_widget import DatabankDownloadWidget

### 2. Download the Databank
Select the cell below and run it.
Enter the Databank ID for the hackathon package.
Enter your email and password for the platform.
Click the **Download** button.

The widget will download and unzip the data right into your current directory. You can monitor the progress in the status output area below the button.

### **Databank ID for PS3: 1ae9a4db-6b53-4a17-8274-fd3818c3f2be**

In [2]:
#uncomment it while running on the cloud
# from databank_download_widget import DatabankDownloadWidget

# # Create an instance of the databank widget
# databank_downloader = DatabankDownloadWidget()

# # Display the widget
# databank_downloader.display()

In [3]:
# =========================
# 2. PATHS & CONFIG
# =========================
import shutil
from pathlib import Path

# Define standard paths
INPUT_DIR = Path("./Claim_Documents")
OUTPUT_DIR = Path("./outputs")
ANNOTATIONS_DIR = OUTPUT_DIR / "annotations"
DEBUG_DIR = OUTPUT_DIR / "rendered_pages"

# ---> THE FIX: Obliterate the old output folder if it exists <---
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

# Create fresh, empty directories
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ANNOTATIONS_DIR.mkdir(parents=True, exist_ok=True)
DEBUG_DIR.mkdir(parents=True, exist_ok=True)

SUPPORTED_EXTS = {".jpg", ".jpeg", ".png", ".pdf"}

print("Environment Setup, Imports, and Output Cleanup Successful.")

CATEGORY_IDS = {
    "C1": "Copy-paste within the same document",
    "C2": "Overwriting on existing text",
    "C3": "Adding new content",
    "C4": "Removing / erasing text or image",
    "C5": "Merging content from different documents",
    "C6": "Watermark removal",
    "C7": "Irregular spacing",
    "C8": "Fully AI-generated document",
    "C9": "Partial AI-generated edits",
    "C10": "No editing / discrepancy found",
}

CATEGORY_ONLY_CLASSES = {"C8", "C10"}

Environment Setup, Imports, and Output Cleanup Successful.


In [4]:
from pathlib import Path

# Define standard paths
INPUT_DIR = Path("./Claim_Documents")
OUTPUT_DIR = Path("./outputs")
ANNOTATIONS_DIR = OUTPUT_DIR / "annotations"
DEBUG_DIR = OUTPUT_DIR / "rendered_pages"

# Create directories
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ANNOTATIONS_DIR.mkdir(parents=True, exist_ok=True)
DEBUG_DIR.mkdir(parents=True, exist_ok=True)

# Fixed variable name to match the onboarding helper logic
SUPPORTED_EXTS = {".jpg", ".jpeg", ".png", ".pdf"}

In [5]:
!optimum-cli export onnx --model dima806/deepfake_vs_real_image_detection --task image-classification hf_model_export/

`torch_dtype` is deprecated! Use `dtype` instead!
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
/home/shivaji/Desktop/PMJAY_AXIOM/.venv/lib/python3.13/site-packages/transformers/models/vit/feature_extraction_vit.py:30: FutureWarning: The class ViTFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use ViTImageProcessor instead.
  warnings.warn(
/home/shivaji/Desktop/PMJAY_AXIOM/.venv/lib/python3.13/site-packages/transformers/models/vit/modeling_vit.py:155: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trac

In [6]:
# =========================
# 4. DATA STRUCTURES
# =========================

@dataclass
class DocumentPage:
    source_link: str
    original_path: str
    original_file_name: str
    page_file_name: str
    page_number: int
    is_pdf: bool
    image_path: Optional[str] = None
    image_width: Optional[int] = None
    image_height: Optional[int] = None


@dataclass
class DetectedRegion:
    x: int
    y: int
    w: int
    h: int
    category_id: str
    type: Optional[str] = None
    stretch_factor: Optional[float] = None
    header_source: Optional[str] = None
    body_source: Optional[str] = None


@dataclass
class PageAnalysisResult:
    source_link: str
    file_name: str
    original_file_name: str
    page_number: int
    predicted_categories: List[str] = field(default_factory=list)
    detected_regions: List[DetectedRegion] = field(default_factory=list)
    notes: Dict[str, Any] = field(default_factory=dict)
    debug: Dict[str, Any] = field(default_factory=dict)

In [7]:

SUPPORTED_IMAGE_EXTS = {".jpg", ".jpeg", ".png"}
SUPPORTED_PDF_EXTS = {".pdf"}

def safe_open_image_size(path):
    try:
        if Image:
            with Image.open(path) as img:
                return img.size
        else:
            img = cv2.imread(str(path))
            if img is not None:
                return img.shape[1], img.shape[0]
    except:
        pass
    return None, None

def render_pdf_to_images(pdf_path, render_dir):
    pages = []
    if not fitz:
        return pages
    try:
        doc = fitz.open(pdf_path)
        for i in range(len(doc)):
            page = doc.load_page(i)
            pix = page.get_pixmap()
            img_path = render_dir / f"{pdf_path.stem}_page_{i+1}.png"
            pix.save(str(img_path))
            pages.append(
                DocumentPage(
                    source_link=str(pdf_path),
                    original_path=str(pdf_path),
                    original_file_name=pdf_path.name,
                    page_file_name=img_path.name,
                    page_number=i+1,
                    is_pdf=True,
                    image_path=str(img_path),
                    image_width=pix.width,
                    image_height=pix.height,
                )
            )
    except:
        pass
    return pages

# =========================
# PIPELINE STAGES (F1-OPTIMIZED WITH NHA STUBS)
# =========================
import os
import cv2
import json
import base64
import numpy as np
import pytesseract
from pathlib import Path
from PIL import Image, ImageChops, ImageEnhance
from typing import List, Dict, Any, Optional

try:
    import onnxruntime as ort
except ImportError:
    ort = None
def build_document_pages(input_dir: Path, render_dir: Path) -> List[DocumentPage]:
    pages: List[DocumentPage] = []
    render_dir.mkdir(parents=True, exist_ok=True)

    for file_path in sorted(input_dir.iterdir()):
        if not file_path.is_file():
            continue

        suffix = file_path.suffix.lower()

        if suffix in SUPPORTED_IMAGE_EXTS:
            width, height = safe_open_image_size(file_path)
            pages.append(
                DocumentPage(
                    source_link=str(file_path),
                    original_path=str(file_path),
                    original_file_name=file_path.name,
                    page_file_name=file_path.name,
                    page_number=1,
                    is_pdf=False,
                    image_path=str(file_path),
                    image_width=width,
                    image_height=height,
                )
            )
        elif suffix in SUPPORTED_PDF_EXTS:
            pages.extend(render_pdf_to_images(file_path, render_dir))

    return pages
# ---------------------------------------------------------
# HELPER MATH ENGINES (FOR PRECISE BOUNDING BOXES)
# ---------------------------------------------------------
def calculate_iou(boxA: List[int], boxB: List[int]) -> float:
    xA, yA = max(boxA[0], boxB[0]), max(boxA[1], boxB[1])
    xB, yB = min(boxA[0] + boxA[2], boxB[0] + boxB[2]), min(boxA[1] + boxA[3], boxB[1] + boxB[3])
    interArea = max(0, xB - xA) * max(0, yB - yA)
    if interArea == 0: return 0.0
    return interArea / float((boxA[2] * boxA[3]) + (boxB[2] * boxB[3]) - interArea)

def run_tier1_ocr_math(img_path: str) -> List[DetectedRegion]:
    regions = []
    img = cv2.imread(img_path)
    if img is None: return regions
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    try:
        d = pytesseract.image_to_data(gray, lang='eng+hin', config=r'--oem 3 --psm 11', output_type=pytesseract.Output.DICT)
    except Exception:
        try:
            d = pytesseract.image_to_data(gray, lang='eng', config=r'--oem 3 --psm 11', output_type=pytesseract.Output.DICT)
        except: return regions
            
    # FALSE POSITIVE FIX 1: Increased confidence threshold from 15 to 45. 
    # This stops Tesseract from thinking random background noise is text.
    boxes = [[d['left'][i], d['top'][i], d['width'][i], d['height'][i]] for i in range(len(d['level'])) if int(d['conf'][i]) > 45 and d['text'][i].strip()]
    
    # C2 Check (Overwriting)
    for i in range(len(boxes)):
        for j in range(i + 1, len(boxes)):
            # FALSE POSITIVE FIX 2: Increased IoU from 0.01 (1%) to 0.15 (15%). 
            # This allows natural cursive handwriting to slightly touch without triggering a forgery flag.
            if calculate_iou(boxes[i], boxes[j]) > 0.15:
                x, y = min(boxes[i][0], boxes[j][0]), min(boxes[i][1], boxes[j][1])
                w, h = max(boxes[i][0]+boxes[i][2], boxes[j][0]+boxes[j][2]) - x, max(boxes[i][1]+boxes[i][3], boxes[j][1]+boxes[j][3]) - y
                regions.append(DetectedRegion(x=int(x), y=int(y), w=int(w), h=int(h), category_id="C2"))

    # C7 Check (Irregular Spacing)
    boxes.sort(key=lambda b: b[1])
    # Pass the image width into the function or just use a safe ratio
    _, img_w = gray.shape
    
    spacing_data = [(boxes[i+1][0] - (boxes[i][0] + boxes[i][2]), boxes[i+1]) for i in range(len(boxes)-1) if abs(boxes[i][1] - boxes[i+1][1]) < 15 and (boxes[i+1][0] - (boxes[i][0] + boxes[i][2])) > 0]
    if spacing_data:
        spaces = [s[0] for s in spacing_data]
        med, std = np.median(spaces), np.std(spaces)
        for space, box in spacing_data:
            # STRICT FIX: Must mathematically deviate, AND space must be less than 40% of the page width 
            # (to prevent flagging natural gaps between table columns)
            if (med > 0 and space > med + (3.0 * std)) and space < (img_w * 0.40):
                regions.append(DetectedRegion(x=int(box[0]-space), y=int(box[1]), w=int(space+box[2]), h=int(box[3]), category_id="C7"))
    return regions

def run_tier2_sift(img_path: str) -> List[DetectedRegion]:
    regions = []
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None: return regions
    kp, des = cv2.SIFT_create().detectAndCompute(img, None)
    if des is not None and len(kp) > 50:
        matches = cv2.FlannBasedMatcher(dict(algorithm=1, trees=5), dict(checks=50)).knnMatch(des, des, k=3)
        good = [m[0] if m[0].queryIdx != m[0].trainIdx else m[1] for m in matches if len(m) >= 2]
        good = [m for m in good if np.linalg.norm(np.array(kp[m.queryIdx].pt) - np.array(kp[m.trainIdx].pt)) > 100 and m.distance < 0.75]
        if len(good) > 15:
            x, y, w, h = cv2.boundingRect(np.float32([kp[m.queryIdx].pt for m in good]).reshape(-1, 2))
            regions.append(DetectedRegion(x=x, y=y, w=w, h=h, category_id="C1"))
    return regions

def run_tier3_ela(img_path: str) -> List[DetectedRegion]:
    regions = []
    try:
        img_bgr = cv2.imread(img_path)
        original = Image.open(img_path).convert('RGB')
        original.save("temp.jpg", 'JPEG', quality=75)
        diff = ImageChops.difference(original, Image.open("temp.jpg"))
        os.remove("temp.jpg")
        
        ela_map = ImageEnhance.Brightness(diff).enhance(255.0 / (max([ex[1] for ex in diff.getextrema()]) or 1))
        _, thresh = cv2.threshold(cv2.cvtColor(np.array(ela_map), cv2.COLOR_RGB2GRAY), 150, 255, cv2.THRESH_BINARY)
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        h, w = gray.shape
        for c in contours:
            x, y, bw, bh = cv2.boundingRect(c)
            # FALSE POSITIVE FIX 4: Increased minimum area from 100 pixels to 400 pixels.
            # This stops tiny specks of dust on the scanner glass from being flagged as a C3 text insertion.
            if bw * bh > 400:
                avg = np.mean(gray[y:y+bh, x:x+bw])
                if bw * bh > (h * w * 0.10) and avg > 190: cat = "C6"
                else: cat = "C4" if avg > 200 else "C3"
                regions.append(DetectedRegion(x=x, y=y, w=bw, h=bh, category_id=cat))
                
    # C5 (Stitched Documents)
        # STRICT FIX: Increased threshold from 8 to 15 to prevent false positives on pages with large dark logos or X-rays
        if abs(np.median(gray[0:h//2, :]) - np.median(gray[h//2:h, :])) > 15:
            regions.append(DetectedRegion(x=0, y=h//2, w=w, h=h//2, category_id="C5"))
    except: pass
    return regions

def run_tier5_red_mask(img_path: str) -> List[DetectedRegion]:
    """
    Tier 5 Scanner: Hunts for solid digital red redaction boxes (C4).
    Uses HSV color space to isolate highly saturated reds drawn over images.
    """
    regions = []
    try:
        img = cv2.imread(img_path)
        if img is None: return regions
        
        # Convert BGR to HSV for precise color isolation
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        
        # In OpenCV, Red wraps around the hue spectrum (0-10 and 160-180).
        # We set Saturation and Value high (>120) to only catch "digital" red, not natural lighting.
        lower_red1 = np.array([0, 120, 120])
        upper_red1 = np.array([10, 255, 255])
        mask1 = cv2.inRange(hsv, lower_red1, upper_red1)
        
        lower_red2 = np.array([160, 120, 120])
        upper_red2 = np.array([180, 255, 255])
        mask2 = cv2.inRange(hsv, lower_red2, upper_red2)
        
        # Combine the two red masks
        full_mask = mask1 + mask2
        
        # Morphological operations to clean up random red noise (like a tiny logo)
        kernel = np.ones((5,5), np.uint8)
        full_mask = cv2.morphologyEx(full_mask, cv2.MORPH_OPEN, kernel)
        full_mask = cv2.dilate(full_mask, kernel, iterations=3)
        
        # Find the physical borders of the red blobs
        contours, _ = cv2.findContours(full_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        h, w = img.shape[:2]
        page_area = h * w
        
        for c in contours:
            bx, by, bw, bh = cv2.boundingRect(c)
            area = bw * bh
            
            # The Box Filter: 
            # We only want MASSIVE red boxes hiding text, not tiny red dots.
            # If the red box is larger than 3000 pixels, it's a redaction mark.
            if area > 3000:
                regions.append(DetectedRegion(
                    x=bx, y=by, w=bw, h=bh, 
                    category_id="C4", 
                    type="erased"
                ))
    except Exception as e:
        pass
        
    return regions

def run_tier5_zero_variance_mask(img_path: str) -> List[DetectedRegion]:
    """
    Tier 5 Scanner: Hunts for solid digital redaction boxes (C4).
    Calculates local pixel variance to find mathematically "flat" areas 
    typical of MS Paint, PDF editors, or WhatsApp markup tools.
    """
    regions = []
    try:
        # Load grayscale, as variance doesn't care about color, just intensity
        img_gray = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img_gray is None: return regions

        # Apply a Blur to smooth out scanner noise, leaving only massive solid blocks intact
        blurred = cv2.GaussianBlur(img_gray, (15, 15), 0)

        # Calculate local standard deviation using OpenCV
        # A perfectly solid color (whiteout, blackbox) will have a standard deviation near 0.
        mean, stddev = cv2.meanStdDev(blurred)
        
        # We need local variance, not global. 
        # Using a rolling Laplacian variance gives us the edge/texture map.
        laplacian = cv2.Laplacian(blurred, cv2.CV_64F)
        variance_map = cv2.convertScaleAbs(laplacian)

        # Threshold the variance map. 
        # We want areas with NO variance (less than 5).
        _, zero_variance_mask = cv2.threshold(variance_map, 5, 255, cv2.THRESH_BINARY_INV)

        # Morphological Operations to clean up the mask
        # 1. Erosion to remove tiny flat spots (like the inside of letters)
        # 2. Dilation to restore the true size of large redaction blocks
        kernel = np.ones((15,15), np.uint8)
        mask_cleaned = cv2.erode(zero_variance_mask, kernel, iterations=1)
        mask_cleaned = cv2.dilate(mask_cleaned, kernel, iterations=3)

        # Find the physical borders of these solid blocks
        contours, _ = cv2.findContours(mask_cleaned, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        h, w = img_gray.shape
        page_area = h * w

        for c in contours:
            bx, by, bw, bh = cv2.boundingRect(c)
            area = bw * bh
            
            # The Box Filter: 
            # We only want MASSIVE redaction blocks.
            # E.g., area > 3000 pixels means it's a deliberate block, not a stray mark.
            # Also ensure it isn't the entire page itself (area < page_area * 0.9)
            if 3000 < area < (page_area * 0.9):
                # Verify that it's actually obscuring the main body of the document, 
                # not just the empty white margins at the edge of the paper.
                if bx > 20 and by > 20 and (bx+bw) < (w-20) and (by+bh) < (h-20):
                    regions.append(DetectedRegion(
                        x=bx, y=by, w=bw, h=bh, 
                        category_id="C4", 
                        type="erased"
                    ))
    except Exception as e:
        print(f"Tier 5 Zero-Variance Error: {e}")
        
    return regions
# ---------------------------------------------------------
# REQUIRED HACKATHON STUBS
# ---------------------------------------------------------


async def run_nha_vision_prompt(image_path: str, prompt_text: str, model_name: str = "bedrock/converse/amazon.nova-lite-v1:0") -> Dict[str, Any]:
    """
    Simulated API call. In a real environment with credentials, this would use aiohttp to hit Amazon Bedrock.
    Because VLMs fail at exact bounding box math, we rely on the OpenCV engines for the actual logic.
    """
    try:
        with open(image_path, "rb") as image_file:
            encoded_string = base64.b64encode(image_file.read()).decode('utf-8')
    except Exception:
        pass
    # Returning a safe dummy structure. OpenCV will overwrite this.
    return {"status": "simulated_success", "predicted_categories": ["C10"]}


def assess_page_quality(page: DocumentPage) -> Dict[str, Any]:
    if not page.image_path: return {"status": "error"}
    img = cv2.imread(page.image_path, cv2.IMREAD_GRAYSCALE)
    if img is None: return {"status": "error"}
    
    lap_var = cv2.Laplacian(img, cv2.CV_64F).var()
    return {
        "laplacian_variance": round(lap_var, 2),
        "is_blurry": bool(lap_var < 100.0),
        "status": "ok"
    }

async def detect_tampering_categories(page: DocumentPage) -> List[str]:
    categories = []
    if not page.image_path: return ["C10"]
    
    # Global AI Checks (C8/C9) using the Pre-trained ViT
    try:
        # 1. FIX: Read as BGR color, not Grayscale, because the ViT expects RGB input
        img_bgr = cv2.imread(page.image_path, cv2.IMREAD_COLOR)
        if img_bgr is None: return ["C10"]
        
        model_path = "./hf_model_export/model.onnx"
        if ort is not None and Path(model_path).exists():
            session = ort.InferenceSession(model_path, providers=['CPUExecutionProvider'])
            input_name = session.get_inputs()[0].name
            
            def check_ai_crop(crop_img_bgr):
                # 2. FIX: Convert BGR to RGB
                rgb_crop = cv2.cvtColor(crop_img_bgr, cv2.COLOR_BGR2RGB)
                
                # 3. FIX: Resize and apply Standard ImageNet Normalization
                resized = cv2.resize(rgb_crop, (224, 224)).astype(np.float32) / 255.0
                mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
                std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
                normalized = (resized - mean) / std
                
                # Transpose to (Channels, Height, Width) and add Batch dimension
                tensor = np.transpose(normalized, (2, 0, 1))
                tensor = np.expand_dims(tensor, axis=0)
                
                # Infer
                logits = session.run(None, {input_name: tensor})[0][0]
                
                # The dima806 model outputs [Real_Logit, Fake_Logit]
                # If Fake (Index 1) > Real (Index 0), flag it.
                if len(logits) > 1:
                    return float(logits[1]) > float(logits[0])
                else:
                    return float(logits) > 0.80

            # Check full image for C8 (Fully AI)
            if check_ai_crop(img_bgr): 
                categories.append("C8")
                
            # Check Quadrants for C9 (Partial AI)
            h, w, _ = img_bgr.shape
            quadrants = [
                img_bgr[0:h//2, 0:w//2], img_bgr[0:h//2, w//2:w], 
                img_bgr[h//2:h, 0:w//2], img_bgr[h//2:h, w//2:w]
            ]
            ai_quadrants_count = sum([1 for q in quadrants if check_ai_crop(q)])
            
            if 0 < ai_quadrants_count < 4:
                categories.append("C9")
                
    except Exception as e:
        print(f"Tier 4 AI Analysis Error on {page.page_file_name}: {e}")
    
    return categories if categories else ["C10"]

async def localize_tampered_regions(page: DocumentPage, predicted_categories: List[str]) -> List[DetectedRegion]:
    regions = []
    if not page.image_path: return regions
    
    # We execute our deterministic CV engines here to guarantee precise YAML bounding boxes.
    regions.extend(run_tier1_ocr_math(page.image_path))
    regions.extend(run_tier2_sift(page.image_path))
    regions.extend(run_tier3_ela(page.image_path))
    # ---> THE NEW RED MASK ENGINE <---
    regions.extend(run_tier5_red_mask(page.image_path))
    # ---> THE NEW ZERO-VARIANCE C4 ENGINE <---
    regions.extend(run_tier5_zero_variance_mask(page.image_path))
    
    return regions

def postprocess_regions(regions: List[DetectedRegion], predicted_categories: List[str], page: DocumentPage) -> List[DetectedRegion]:
    if not regions: return []
    img_w = page.image_width or float('inf')
    img_h = page.image_height or float('inf')
    
    valid = []
    for r in regions:
        x1, y1 = max(0, r.x), max(0, r.y)
        w, h = min(img_w, r.x + r.w) - x1, min(img_h, r.y + r.h) - y1
        if w > 0 and h > 0:
            r.x, r.y, r.w, r.h = int(x1), int(y1), int(w), int(h)
            valid.append(r)

    merged = []
    grouped = {}
    for r in valid: grouped.setdefault(r.category_id, []).append(r)

    for cat_id, boxes in grouped.items():
        active = boxes.copy()
        while active:
            curr = active.pop(0)
            was_merged = False
            for i, other in enumerate(active):
                if calculate_iou([curr.x, curr.y, curr.w, curr.h], [other.x, other.y, other.w, other.h]) > 0.10:
                    nx, ny = min(curr.x, other.x), min(curr.y, other.y)
                    curr.w, curr.h = max(curr.x + curr.w, other.x + other.w) - nx, max(curr.y + curr.h, other.y + other.h) - ny
                    curr.x, curr.y = nx, ny
                    active.pop(i)
                    active.append(curr)
                    was_merged = True
                    break
            if not was_merged: merged.append(curr)
    
    # Remove strict duplicates
    final, seen = [], set()
    for r in merged:
        sig = (r.x, r.y, r.w, r.h, r.category_id)
        if sig not in seen:
            seen.add(sig)
            final.append(r)
    return final

def enrich_region_metadata(regions: List[DetectedRegion], predicted_categories: List[str], page: DocumentPage) -> List[DetectedRegion]:
    """
    Strictly aligns the YAML metadata keys with the NHA Hackathon guidelines.
    """
    for r in regions:
        if r.category_id == "C3": 
            r.type = "text" # As per Figure 3.3
        elif r.category_id == "C4": 
            r.type = "erased" # As per Figure 4.3
        elif r.category_id == "C7": 
            r.type = "irregular_spacing"
            r.stretch_factor = round(r.stretch_factor, 2) if r.stretch_factor else 1.25
        elif r.category_id == "C5":
            # MANDATORY GUIDELINE RULE: Use the keyword 'other' for sources
            r.type = "body"
            r.header_source = "other"
            r.body_source = "other"
        elif r.category_id == "C9": 
            r.type = "amount" # Fallback C9 type as per Figure 9.3
            
    return regions

def validate_prediction(page: DocumentPage, predicted_categories: List[str], regions: List[DetectedRegion]) -> Dict[str, Any]:
    VALID_CATS = {"C1", "C2", "C3", "C4", "C5", "C6", "C7", "C8", "C9", "C10"}
    CAT_ONLY = {"C8", "C10"}
    
    region_cats = set(r.category_id for r in regions)
    all_cats = set(predicted_categories).union(region_cats).intersection(VALID_CATS)
    
    if len(all_cats) > 1 and "C10" in all_cats: all_cats.remove("C10")
    if not all_cats: all_cats.add("C10")
    
    # Sort: Place global C8 first, then sort remaining by area size
    cat_areas = {cat: sum(r.w * r.h for r in regions if r.category_id == cat) for cat in all_cats}
    localized = sorted([c for c in all_cats if c not in CAT_ONLY], key=lambda c: cat_areas.get(c, 0), reverse=True)
    
    final_ordered = []
    if "C8" in all_cats: final_ordered.append("C8")
    final_ordered.extend(localized)
    if "C10" in all_cats and "C10" not in final_ordered: final_ordered.append("C10")

    return {
        "valid": True,
        "final_categories": final_ordered,
        "dropped_categories": [],
        "notes": "Validation complete."
    }

def build_page_analysis_result(page: DocumentPage, predicted_categories: List[str], regions: List[DetectedRegion], quality_summary: Optional[Dict[str, Any]] = None, validation: Optional[Dict[str, Any]] = None) -> PageAnalysisResult:
    final_cats = validation.get("final_categories", ["C10"]) if validation else predicted_categories
    return PageAnalysisResult(
        source_link=page.source_link,
        file_name=page.page_file_name,
        original_file_name=page.original_file_name,
        page_number=page.page_number,
        predicted_categories=final_cats,
        detected_regions=regions,
        notes={"pipeline": "cv_hybrid_engine"},
        debug={"quality": quality_summary, "validation": validation}
    )

## Output Builders

This section should follow the guideline requirements:
- JSON rows contain `link`, `file_name`, and `Category_ID`.
- YAML files are emitted per page using the page-style file name.
- Categories are joined with `||` in the JSON output when multiple apply.
- C8 and C10 do not require YAML. 

In [8]:
# =========================
# 7. OUTPUT / EXPORT HELPERS 
# =========================

def normalize_category_list(predicted_categories: List[str]) -> List[str]:
    valid = [c for c in predicted_categories if c in CATEGORY_IDS]
    seen = set()
    ordered = []
    for c in valid:
        if c not in seen:
            ordered.append(c)
            seen.add(c)
    return ordered


def build_json_row(result: PageAnalysisResult) -> Dict[str, Any]:
    categories = normalize_category_list(result.predicted_categories)
    category_string = "||".join(categories) if categories else "C10"

    return {
        "link": result.source_link,
        "file_name": result.file_name,
        "Category_ID": category_string,
    }


def detected_region_to_yaml_item(region: DetectedRegion) -> Dict[str, Any]:
    item: Dict[str, Any] = {
        "h": int(region.h),
        "w": int(region.w),
        "x": int(region.x),
        "y": int(region.y),
        "category_id": region.category_id,
    }

    if region.type is not None:
        item["type"] = region.type
    if region.stretch_factor is not None:
        item["stretch_factor"] = region.stretch_factor
    if region.header_source is not None:
        item["header_source"] = region.header_source
    if region.body_source is not None:
        item["body_source"] = region.body_source

    return item


def yaml_required_for_result(result: PageAnalysisResult) -> bool:
    categories = set(normalize_category_list(result.predicted_categories))
    if not categories:
        return False
    return not categories.issubset(CATEGORY_ONLY_CLASSES)


def yaml_name_for_page_file(page_file_name: str) -> str:
    return f"{Path(page_file_name).stem}.yaml"


def write_yaml_for_result(result: PageAnalysisResult, annotations_dir: Path) -> Optional[Path]:
    if not yaml_required_for_result(result):
        return None

    if yaml is None:
        raise ImportError("PyYAML is required to write annotation YAML files.")

    yaml_path = annotations_dir / yaml_name_for_page_file(result.file_name)
    yaml_items = [detected_region_to_yaml_item(region) for region in result.detected_regions]

    with open(yaml_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(yaml_items, f, sort_keys=False, allow_unicode=True)

    return yaml_path


def write_submission_json(results: List[PageAnalysisResult], output_dir: Path) -> Path:
    json_rows = [build_json_row(r) for r in results]
    json_path = output_dir / "submission.json"

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(json_rows, f, indent=2, ensure_ascii=False)

    return json_path


def write_optional_excel_summary(results: List[PageAnalysisResult], output_dir: Path) -> Optional[Path]:
    """
    DISABLED: The NHA guidelines ONLY require submission.json and the YAML files.
    Generating an Excel file causes openpyxl dependency crashes in the cloud environment.
    Returning None ensures a perfectly safe, crash-free run.
    """
    return None


def export_all_outputs(results: List[PageAnalysisResult], output_dir: Path, annotations_dir: Path) -> Dict[str, Any]:
    annotation_paths = []
    for result in results:
        yaml_path = write_yaml_for_result(result, annotations_dir)
        if yaml_path is not None:
            annotation_paths.append(str(yaml_path))

    json_path = write_submission_json(results, output_dir)
    excel_path = write_optional_excel_summary(results, output_dir)

    return {
        "json_path": str(json_path),
        "yaml_paths": annotation_paths,
        "excel_preview_path": str(excel_path) if excel_path else None,
        "total_results": len(results),
    }

## Main Runner

The runner is fully implemented. It:
- onboards files,
- expands PDFs to page images,
- runs page-wise pipeline stages,
- creates final page results,
- exports JSON and YAML outputs.

The actual detection stages remain empty and will fall back to scaffold behavior until implemented.

In [9]:
# =========================
# 9. MAIN RUNNER (FILLED)
# =========================

async def run_ps3_pipeline(
    input_dir: Path = INPUT_DIR,
    output_dir: Path = OUTPUT_DIR,
    annotations_dir: Path = ANNOTATIONS_DIR,
    debug_dir: Path = DEBUG_DIR,
) -> Dict[str, Any]:
    render_dir = debug_dir / "rendered_pages"
    pages = build_document_pages(input_dir, render_dir)

    results: List[PageAnalysisResult] = []

    # THE FIX: Added enumerate(pages) to define idx, and added live tracking prints
    for idx, page in enumerate(pages):
        print(f"\n[{idx + 1}/{len(pages)}] Processing: {page.page_file_name}...")
        
        # Quality Assessment
        try:
            quality_summary = assess_page_quality(page)
            if quality_summary is None:
                quality_summary = {"status": "error"}
        except Exception as e:
            print(f"   [!] Error in assess_page_quality: {e}")
            quality_summary = {"status": "error"}

        # Categorization (Tier 4)
        try:
            predicted_categories = await detect_tampering_categories(page)
            if predicted_categories is None:
                predicted_categories = ["C10"]
        except Exception as e:
            print(f"   [!] Error in detect_tampering_categories: {e}")
            predicted_categories = ["C10"]

        predicted_categories = normalize_category_list(predicted_categories) or ["C10"]

        # Localization (Tiers 1, 2, 3)
        try:
            regions = await localize_tampered_regions(page, predicted_categories)
            if regions is None:
                regions = []
        except Exception as e:
            print(f"   [!] Error in localize_tampered_regions: {e}")
            regions = []

        # Postprocessing
        try:
            regions = postprocess_regions(regions, predicted_categories, page)
            if regions is None:
                regions = []
        except Exception as e:
            print(f"   [!] Error in postprocess_regions: {e}")
            regions = []

        # Enrichment
        try:
            regions = enrich_region_metadata(regions, predicted_categories, page)
            if regions is None:
                regions = []
        except Exception as e:
            print(f"   [!] Error in enrich_region_metadata: {e}")
            regions = []

        # Validation
        try:
            validation = validate_prediction(page, predicted_categories, regions)
            if validation is None:
                validation = {"final_categories": predicted_categories or ["C10"]}
        except Exception as e:
            print(f"   [!] Error in validate_prediction: {e}")
            validation = {"final_categories": predicted_categories or ["C10"]}

        # Build final result
        try:
            result = build_page_analysis_result(
                page=page,
                predicted_categories=predicted_categories,
                regions=regions,
                quality_summary=quality_summary,
                validation=validation,
            )
        except Exception as e:
            print(f"   [!] Error in build_page_analysis_result: {e}")
            result = PageAnalysisResult(
                source_link=page.source_link,
                file_name=page.page_file_name,
                original_file_name=page.original_file_name,
                page_number=page.page_number,
                predicted_categories=["C10"],
            )

        results.append(result)
        
        # --- LIVE TRACKING OUTPUT ---
        final_cats = result.predicted_categories
        cat_string = "||".join(final_cats) if final_cats else "C10"
        
        # Check if YAML is needed (C8 and C10 skip YAML generation)
        if set(final_cats).issubset({"C8", "C10"}):
            yaml_name = "N/A (Category Only - No YAML required)"
        else:
            yaml_name = f"{Path(page.page_file_name).stem}.yaml"
            
        print(f"   -> Categories: {cat_string}")
        print(f"   -> Bounding Boxes: {len(result.detected_regions)}")
        print(f"   -> YAML Target: {yaml_name}")

    print("\n" + "="*50)
    print("All pages processed. Exporting final JSON and YAMLs to disk...")
    export_info = export_all_outputs(results, output_dir, annotations_dir)

    return {
        "pages": pages,
        "results": results,
        "export_info": export_info,
    }


## Display Helpers

In [10]:
# =========================
# 10. DISPLAY FINAL OUTPUTS (FILLED)
# =========================

def display_final_outputs(run_output: Dict[str, Any]) -> None:
    results: List[PageAnalysisResult] = run_output.get("results", [])
    export_info: Dict[str, Any] = run_output.get("export_info", {})

    print("=" * 80)
    print("PS3 PIPELINE RUN SUMMARY")
    print("=" * 80)
    print(f"Total page-level entries processed: {len(results)}")
    print(f"Submission JSON: {export_info.get('json_path')}")
    print(f"YAML files written: {len(export_info.get('yaml_paths', []))}")
    if export_info.get("excel_preview_path"):
        print(f"Excel preview: {export_info.get('excel_preview_path')}")
    print()

    preview_rows = [build_json_row(r) for r in results[:10]]
    if pd is not None and preview_rows:
        display(pd.DataFrame(preview_rows))
    else:
        print("Preview JSON rows:")
        print(json.dumps(preview_rows, indent=2))

    if export_info.get("yaml_paths"):
        print()
        print("Sample YAML files:")
        for yp in export_info["yaml_paths"][:10]:
            print("-", yp)

## Execution Cell



In [11]:
# !pip install openpyxl


In [12]:
# =========================
# 11. EXECUTE
# =========================

run_output = await run_ps3_pipeline()
display_final_outputs(run_output)


[1/829] Processing: 000001__PMJAY_UK_S_2025_R3_2026032210013943__SS_page_1.png...
   -> Categories: C4||C7
   -> Bounding Boxes: 5
   -> YAML Target: 000001__PMJAY_UK_S_2025_R3_2026032210013943__SS_page_1.yaml

[2/829] Processing: 000001__PMJAY_UK_S_2025_R3_2026032210013943__SS_page_2.png...
   -> Categories: C4||C7||C2
   -> Bounding Boxes: 4
   -> YAML Target: 000001__PMJAY_UK_S_2025_R3_2026032210013943__SS_page_2.yaml

[3/829] Processing: 000002__PMJAY_UK_S_2025_R3_2026032210013943__img20260322_15254944.jpg...
   -> Categories: C2
   -> Bounding Boxes: 2
   -> YAML Target: 000002__PMJAY_UK_S_2025_R3_2026032210013943__img20260322_15254944.yaml

[4/829] Processing: 000003__PMJAY_UK_S_2025_R3_2026032210013943__X.jpg...
   -> Categories: C8||C5||C9
   -> Bounding Boxes: 1
   -> YAML Target: 000003__PMJAY_UK_S_2025_R3_2026032210013943__X.yaml

[5/829] Processing: 000004__PMJAY_UK_S_2025_R3_2026032210013943__img20260322_15251997.jpg...
   -> Categories: C10
   -> Bounding Boxes: 0
   -> 

,link,file_name,Category_ID
0,Claim_Documents/000001__PMJAY_UK_S_2025_R3_202...,000001__PMJAY_UK_S_2025_R3_2026032210013943__S...,C4||C7
1,Claim_Documents/000001__PMJAY_UK_S_2025_R3_202...,000001__PMJAY_UK_S_2025_R3_2026032210013943__S...,C4||C7||C2
2,Claim_Documents/000002__PMJAY_UK_S_2025_R3_202...,000002__PMJAY_UK_S_2025_R3_2026032210013943__i...,C2
3,Claim_Documents/000003__PMJAY_UK_S_2025_R3_202...,000003__PMJAY_UK_S_2025_R3_2026032210013943__X...,C8||C5||C9
4,Claim_Documents/000004__PMJAY_UK_S_2025_R3_202...,000004__PMJAY_UK_S_2025_R3_2026032210013943__i...,C10
5,Claim_Documents/000005__PMJAY_UK_S_2025_R3_202...,000005__PMJAY_UK_S_2025_R3_2026032210013943__i...,C4||C7||C2
6,Claim_Documents/000006__PMJAY_UK_S_2025_R3_202...,000006__PMJAY_UK_S_2025_R3_2026032210013943__i...,C4
7,Claim_Documents/000007__PMJAY_UK_S_2025_R3_202...,000007__PMJAY_UK_S_2025_R3_2026032210013943__i...,C5
8,Claim_Documents/000008__PMJAY_UK_S_2025_R3_202...,000008__PMJAY_UK_S_2025_R3_2026032210013943__X...,C8||C5
9,Claim_Documents/000010__PMJAY_UK_S_2025_R3_202...,000010__PMJAY_UK_S_2025_R3_2026032210013943__1...,C10



Sample YAML files:
- outputs/annotations/000001__PMJAY_UK_S_2025_R3_2026032210013943__SS_page_1.yaml
- outputs/annotations/000001__PMJAY_UK_S_2025_R3_2026032210013943__SS_page_2.yaml
- outputs/annotations/000002__PMJAY_UK_S_2025_R3_2026032210013943__img20260322_15254944.yaml
- outputs/annotations/000003__PMJAY_UK_S_2025_R3_2026032210013943__X.yaml
- outputs/annotations/000005__PMJAY_UK_S_2025_R3_2026032210013943__img20260328_15531199.yaml
- outputs/annotations/000006__PMJAY_UK_S_2025_R3_2026032210013943__img20260328_15534909.yaml
- outputs/annotations/000007__PMJAY_UK_S_2025_R3_2026032210013943__img20250419_09093982.yaml
- outputs/annotations/000008__PMJAY_UK_S_2025_R3_2026032210013943__XXX.yaml
- outputs/annotations/000011__PMJAY_UK_S_2025_R3_2026032210013943__img20260328_15531199.yaml
- outputs/annotations/000013__PMJAY_UK_S_2025_R3_2026032210013943__111.yaml
